## Cell 1 — Install dependencies

Run this first in Colab.


In [ ]:
!pip -q install openai python-dotenv


## Cell 2 — Imports and environment loading

`load_dotenv()` reads your `.env` file and loads variables like:

- `GROQ_API_KEY`
- `OPENROUTER_API_KEY`
- `OPENAI_API_KEY`

Only one is needed.


In [ ]:
import json
import os
from typing import Any

from dotenv import load_dotenv

load_dotenv()

SEPARATOR = "<---------------->"
DOUBLE_LINE = "=" * 70


## Cell 3 — Sample data and tools

A **tool** is just a normal Python function.

The model does **not** run Python itself.

The model only says:

- which function to call
- with what arguments

Then our Python code runs the function.


In [ ]:
SAMPLE_WEATHER = {
    "tokyo": {"celsius": 22, "conditions": "partly cloudy"},
    "delhi": {"celsius": 34, "conditions": "clear skies"},
    "london": {"celsius": 15, "conditions": "light rain"},
}


def get_weather(city: str) -> str:
    # Check the local sample dictionary and return a readable string.
    data = SAMPLE_WEATHER.get(city.lower())
    if data is None:
        return f"No weather data for {city!r}."
    return f"{city.title()}: {data['celsius']}C, {data['conditions']}"


def get_currency(from_currency: str, to_currency: str) -> str:
    # Placeholder currency tool.
    # It exists in the code, but it is not exposed to the model in TOOL_SCHEMAS.
    return f"The exchange rate from {from_currency} to {to_currency} is 1.0."


## Cell 4 — Tool schemas

The schema is what the model sees so it can decide:

- whether a tool exists
- what arguments that tool expects
- how to format the tool call

Only `get_weather` is exposed to the model here, matching your original flow.


In [ ]:
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            },
        },
    }
]

TOOLS_BY_NAME = {
    "get_weather": get_weather,
    "get_currency": get_currency,
}


## Cell 5 — Provider selection

This keeps the same logic as your original code:

1. Try Groq
2. Else try OpenRouter
3. Else try OpenAI
4. Else raise an error

All three are OpenAI-compatible, so we use the same client class.


In [ ]:
os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"  # Replace with your actual Groq API key

In [ ]:
def get_client_and_model():
    # Return an OpenAI-compatible client and the model name.
    # Same provider picker as the original script.
    from openai import OpenAI

    if os.environ.get("GROQ_API_KEY"):
        return (
            OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1"),
            "llama-3.3-70b-versatile",
        )

    if os.environ.get("OPENROUTER_API_KEY"):
        return (
            OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1"),
            "openrouter/free",
        )

    if os.environ.get("OPENAI_API_KEY"):
        return OpenAI(api_key=os.environ["OPENAI_API_KEY"]), "gpt-4o-mini"

    raise RuntimeError(
        "No OpenAI-compatible key found. Set one of GROQ_API_KEY, "
        "OPENROUTER_API_KEY, or OPENAI_API_KEY in your .env file."
    )


## Cell 6 — Helper functions for very clear printing

These functions do **not** change behavior.

They only show what is happening.


In [ ]:
def pretty_json(value: Any) -> str:
    return json.dumps(value, indent=2, ensure_ascii=False)


def print_title(title: str) -> None:
    print("\n" + DOUBLE_LINE)
    print(title)
    print(DOUBLE_LINE)


def print_separator() -> None:
    print(SEPARATOR)


def print_memory(messages: list[dict], heading: str = "Conversation memory") -> None:
    print_title(heading)
    print(f"Memory contains {len(messages)} message(s)\n")

    for index, msg in enumerate(messages):
        print(f"[{index}] ROLE: {msg.get('role')}")
        if "content" in msg and msg["content"] is not None:
            print("CONTENT:")
            print(msg["content"])
        else:
            print("CONTENT: None")

        if msg.get("tool_calls"):
            print("TOOL CALLS:")
            for i, call in enumerate(msg["tool_calls"], start=1):
                fn = call["function"]["name"]
                args = call["function"]["arguments"]
                print(f"  {i}. id: {call['id']}")
                print(f"     name: {fn}")
                print(f"     arguments: {args}")

        if msg.get("tool_call_id"):
            print(f"TOOL CALL ID: {msg['tool_call_id']}")

        print("-" * 60)


def print_tool_schemas(tool_schemas: list[dict]) -> None:
    print_title("Available tool schemas")
    for i, schema in enumerate(tool_schemas, start=1):
        fn = schema["function"]
        print(f"{i}. NAME: {fn['name']}")
        print(f"   DESCRIPTION: {fn['description']}")
        print("   ARGUMENTS:")
        print(pretty_json(fn["parameters"]))
        print("-" * 60)


def print_raw_llm_message(message: Any) -> None:
    print_title("Raw LLM response")
    print("assistant.content:")
    print(message.content)
    print("\nassistant.tool_calls:")
    if not message.tool_calls:
        print("None")
        return

    for i, call in enumerate(message.tool_calls, start=1):
        print(f"\nTool call #{i}")
        print(f"  id: {call.id}")
        print(f"  name: {call.function.name}")
        print(f"  arguments (raw): {call.function.arguments}")


def print_tool_execution(tool_name: str, arguments: dict, result: Any) -> None:
    print_title("Tool execution")
    print(f"Requested tool: {tool_name}")
    print("Parsed arguments:")
    print(pretty_json(arguments))
    print("\nResult returned by Python:")
    print(result)


## Cell 7 — The agent loop

This is the heart of the notebook.

The loop is simple:

1. Send current memory to the model
2. See whether the model wants a tool
3. If yes, run the tool and add the result to memory
4. If no, return the final answer
5. Stop after `max_turns` so the loop cannot run forever

The model does **not** execute tools.

The model only requests them.


In [ ]:
def run_agent(messages: list[dict], max_turns: int = 4, verbose: bool = True) -> str:
    # The agent loop.
    #
    # Each turn:
    # - call the model with the tool schema attached
    # - if the model replies with tool_calls, execute each one
    # - append tool results back into memory
    # - call the model again
    # - if the model replies with plain text only, stop and return it
    #
    # max_turns is a safety limit so the loop cannot continue forever.
    client, model = get_client_and_model()

    for turn_number in range(1, max_turns + 1):
        if verbose:
            print_separator()
            print_title(f"AGENT LOOP TURN {turn_number}")
            print_memory(messages, "Memory BEFORE API call")
            print_tool_schemas(TOOL_SCHEMAS)
            print_title("Sending everything to the model")
            print(f"Model: {model}")
            print("What the model receives:")
            print("- conversation memory")
            print("- tool schemas")
            print("- max_tokens setting")
            print("- the current instruction context")
            print("\nImportant: the model does NOT run Python code.")
            print("It only decides whether it needs a tool or can answer directly.")
            print_separator()

        response = client.chat.completions.create(
            model=model,
            max_tokens=300,
            messages=messages,
            tools=TOOL_SCHEMAS,
        )
        message = response.choices[0].message

        if verbose:
            print_raw_llm_message(message)

        # If the model did not request a tool, we treat this as the final answer.
        if not message.tool_calls:
            messages.append({"role": "assistant", "content": message.content})

            if verbose:
                print_title("No tool calls detected")
                print("The model answered directly, so the loop stops here.")
                print_memory(messages, "Memory AFTER final assistant answer")
                print_separator()

            return message.content

        # The model asked for one or more tools.
        messages.append(
            {
                "role": "assistant",
                "content": message.content,
                "tool_calls": [
                    {
                        "id": call.id,
                        "type": "function",
                        "function": {
                            "name": call.function.name,
                            "arguments": call.function.arguments,
                        },
                    }
                    for call in message.tool_calls
                ],
            }
        )

        if verbose:
            print_title("Assistant requested tool call(s)")
            print("The assistant message has been added to memory.")
            print("Now Python will inspect each requested tool call and execute it.")
            print_memory(messages, "Memory AFTER assistant tool request")

        for call in message.tool_calls:
            tool_name = call.function.name
            raw_arguments = call.function.arguments
            arguments = json.loads(raw_arguments)

            if verbose:
                print_title("Processing one tool call")
                print(f"Tool call id: {call.id}")
                print(f"Tool name   : {tool_name}")
                print(f"Raw args    : {raw_arguments}")
                print("Parsed args  :")
                print(pretty_json(arguments))
                print("\nLooking up the tool in TOOLS_BY_NAME...")
                print(f"Available tools: {list(TOOLS_BY_NAME.keys())}")

            tool_function = TOOLS_BY_NAME[tool_name]
            result = tool_function(**arguments)

            if verbose:
                print_tool_execution(tool_name, arguments, result)

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": str(result),
                }
            )

            if verbose:
                print_title("Tool result added to memory")
                print("The tool output has now been stored as a TOOL message.")
                print_memory(messages, "Memory AFTER tool result")
                print("Why do we loop again?")
                print("Because the model has the tool result only after this step.")
                print("Now it can read the result and produce the final natural-language answer.")
                print_separator()

    return "Reached max_turns without a final answer."


## Cell 8 — Minimal chat loop

This is a simple terminal-style REPL.

The `conversation_memory` list is the entire memory for the session.

Every new user message is appended to that list, and the agent sees the full history on the next turn.


In [ ]:
def chat() -> None:
    # Minimal interactive chat loop.
    #
    # conversation_memory is the agent's entire memory:
    # a plain list of messages that grows after every user turn.
    conversation_memory: list[dict] = []
    print("Type a question (Ctrl+C to quit).")
    print("Remember: the agent will print each step very verbosely.\n")

    while True:
        try:
            user_input = input("You: ")
        except (KeyboardInterrupt, EOFError):
            print("\nExiting.")
            break

        conversation_memory.append({"role": "user", "content": user_input})

        print_separator()
        print_title("New user message appended to memory")
        print_memory(conversation_memory, "Memory BEFORE running the agent")

        answer = run_agent(conversation_memory)
        print("\nAgent final answer:")
        print(answer)
        print()

    print("Goodbye!")
    print("\nFinal conversation memory:")
    print(conversation_memory)


## Cell 9 — Optional quick test

You can run a single turn without the interactive chat loop.

Example:

- ask about Delhi
- the model should request `get_weather`
- Python executes the tool
- the model gets the result and answers


In [ ]:
# Uncomment this if you want to test a single question immediately.

conversation_memory = [{"role": "user", "content": "What is the weather in Delhi?"}]
answer = run_agent(conversation_memory, max_turns=4, verbose=True)
print("\nFINAL ANSWER:")
print(answer)


<---------------->

AGENT LOOP TURN 1

Memory BEFORE API call
Memory contains 1 message(s)

[0] ROLE: user
CONTENT:
What is the weather in Delhi?
------------------------------------------------------------

Available tool schemas
1. NAME: get_weather
   DESCRIPTION: Get the current weather for a city.
   ARGUMENTS:
{
  "type": "object",
  "properties": {
    "city": {
      "type": "string"
    }
  },
  "required": [
    "city"
  ]
}
------------------------------------------------------------

Sending everything to the model
Model: llama-3.3-70b-versatile
What the model receives:
- conversation memory
- tool schemas
- max_tokens setting
- the current instruction context

Important: the model does NOT run Python code.
It only decides whether it needs a tool or can answer directly.
<---------------->

Raw LLM response
assistant.content:
None

assistant.tool_calls:

Tool call #1
  id: 71smjvm07
  name: get_weather
  arguments (raw): {"city":"Delhi"}

Assistant requested tool call(s)
The

## Cell 10 — Start the chat loop

Uncomment the line below when you are ready to try it in Colab.

For a live demo, use a question like:

- `What is the weather in Delhi?`
- `What is the weather in Tokyo?`

That will clearly show the tool-calling loop.


In [ ]:
chat()


Type a question (Ctrl+C to quit).
Remember: the agent will print each step very verbosely.

You: Hi I am Mayank.
<---------------->

New user message appended to memory

Memory BEFORE running the agent
Memory contains 1 message(s)

[0] ROLE: user
CONTENT:
Hi I am Mayank.
------------------------------------------------------------
<---------------->

AGENT LOOP TURN 1

Memory BEFORE API call
Memory contains 1 message(s)

[0] ROLE: user
CONTENT:
Hi I am Mayank.
------------------------------------------------------------

Available tool schemas
1. NAME: get_weather
   DESCRIPTION: Get the current weather for a city.
   ARGUMENTS:
{
  "type": "object",
  "properties": {
    "city": {
      "type": "string"
    }
  },
  "required": [
    "city"
  ]
}
------------------------------------------------------------

Sending everything to the model
Model: llama-3.3-70b-versatile
What the model receives:
- conversation memory
- tool schemas
- max_tokens setting
- the current instruction context
